# Chapter 21 — ICL Training Dynamics

Reproduces:
- Figure 21.1: Training curves on three priors (loss + grad norm + LR).
- Figure 21.2: Learning-rate sensitivity sweep.
- Figure 21.3: Failure mode — too-narrow prior produces a degenerate fit.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tabkernels.priors import SCMPrior, SCMConfig, ARFPrior, MLPSCMPrior
from tabkernels.training import ICLTrainer

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## A small two-layer transformer

The same architecture as Chapter 19; we want to track *training dynamics*,
not architecture comparisons.


In [ ]:
class TinyPFN(nn.Module):
    def __init__(self, d_in, d_emb=48, n_heads=4, n_layers=2, d_ff=96):
        super().__init__()
        self.x_embed = nn.Linear(d_in, d_emb)
        self.y_embed = nn.Linear(1, d_emb)
        self.q_marker = nn.Parameter(torch.randn(d_emb) * 0.02)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'ln1': nn.LayerNorm(d_emb),
                'attn': nn.MultiheadAttention(d_emb, n_heads, batch_first=True),
                'ln2': nn.LayerNorm(d_emb),
                'mlp': nn.Sequential(nn.Linear(d_emb, d_ff), nn.GELU(),
                                     nn.Linear(d_ff, d_emb)),
            }) for _ in range(n_layers)
        ])
        self.out_ln = nn.LayerNorm(d_emb)
        self.head = nn.Linear(d_emb, 1)

    def forward(self, X_q, X_ctx, y_ctx):
        ctx = self.x_embed(X_ctx) + self.y_embed(y_ctx.unsqueeze(-1))
        q = self.x_embed(X_q) + self.q_marker
        seq = torch.cat([ctx, q], dim=0).unsqueeze(0)
        nc = X_ctx.shape[0]
        for layer in self.layers:
            n = layer['ln1'](seq)
            a, _ = layer['attn'](n, n, n, need_weights=False)
            seq = seq + a
            seq = seq + layer['mlp'](layer['ln2'](seq))
        return self.head(self.out_ln(seq[0, nc:])).squeeze(-1)


def make_model(d=4, seed=0):
    torch.manual_seed(seed)
    return TinyPFN(d_in=d, d_emb=48, n_heads=4, n_layers=2, d_ff=96)


## Figure 21.1: Training curves on three priors

For each of three priors (SCM linear, SCM-MLP, MLP-SCM hybrid), plot the loss
trajectory, gradient norm, and learning rate together. The shape of each
curve depends on the prior — harder priors plateau higher and have noisier
gradient norms.


In [ ]:
priors_for_plot = {
    'SCM-linear': SCMPrior(SCMConfig(structural='linear', edge_prob=0.5, noise_scale=0.3)),
    'SCM-MLP':    SCMPrior(SCMConfig(structural='mlp', edge_prob=0.5, noise_scale=0.3)),
}
# MLP-SCM with ARF features.
torch.manual_seed(7)
n_corpus = 400; D = 4
mode_centres = torch.tensor([[2.0, 0.0, 0.0, 0.0], [0.0, 2.0, -2.0, 0.0],
                              [-2.0, -2.0, 1.0, 1.0]])
modes = torch.randint(3, (n_corpus,))
seed_corpus = mode_centres[modes] + 0.5 * torch.randn(n_corpus, D)
arf = ARFPrior(corpus=seed_corpus, max_leaves=24)
priors_for_plot['MLP-SCM'] = MLPSCMPrior(feature_prior=arf, hidden=12, depth=2)

diags = {}
for name, p in priors_for_plot.items():
    model = make_model(d=D, seed=42)
    trainer = ICLTrainer(prior=p, model=model, n_steps=400, n_ctx=48,
                         n_query=24, d=D, lr=3e-3, warmup_steps=30,
                         grad_clip=1.0, eval_every=40, seed=1)
    diags[name] = trainer.train()
    print(f'  {name:<11s}: train end {diags[name].losses[-1]:.4f}, eval end {diags[name].eval_losses[-1]:.4f}')

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
for name, diag in diags.items():
    axes[0].plot(diag.losses, alpha=0.45, label=name)
    axes[1].plot(diag.grad_norms, alpha=0.45, label=name)
axes[2].plot(diags[list(diags)[0]].lrs, color='black')
axes[0].set_ylabel('train loss'); axes[0].set_yscale('log'); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].set_ylabel('grad norm'); axes[1].set_yscale('log'); axes[1].grid(alpha=0.3)
axes[2].set_ylabel('lr'); axes[2].set_xlabel('step'); axes[2].grid(alpha=0.3)
axes[0].set_title('Figure 21.1: ICL training curves on three priors')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_21_01_training_curves.pdf', bbox_inches='tight')
plt.show()


## Figure 21.2: Learning-rate sensitivity

Sweep peak learning rate over four values on the SCM-MLP prior. Too-low rates
under-fit; too-high rates oscillate or diverge. The sweet spot near $3 \times 10^{-3}$
matches what the literature reports for tabular-PFN-class models.


In [ ]:
prior = priors_for_plot['SCM-MLP']
lrs = [1e-4, 1e-3, 3e-3, 1e-2]
sweep = {}
for lr in lrs:
    model = make_model(d=D, seed=42)
    trainer = ICLTrainer(prior=prior, model=model, n_steps=300, n_ctx=48,
                         n_query=24, d=D, lr=lr, warmup_steps=30,
                         grad_clip=1.0, eval_every=0, seed=2)
    sweep[lr] = trainer.train()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for lr, diag in sweep.items():
    ax.plot(diag.losses, label=f'lr = {lr:.0e}', alpha=0.7)
ax.set_xlabel('step'); ax.set_ylabel('train loss')
ax.set_title('Figure 21.2: Learning-rate sweep on SCM-MLP prior')
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_21_02_lr_sweep.pdf', bbox_inches='tight')
plt.show()


## Figure 21.3: Failure mode — over-narrow prior

When the training prior is so narrow that every task uses the same
labelling function, the model can memorise that function and ignore the
context entirely. Held-out evaluation on a *broader* prior (with task-varying
labels) then exposes the failure: the model has not learned ICL, only
amortised one specific task.


In [ ]:
from tabkernels.core.base import Prior


class FixedWPrior(Prior):
    """All tasks use the same fixed regression weights."""
    def __init__(self, w, sigma_eps=0.1):
        self.w = w
        self.sigma_eps = sigma_eps
    def sample_episode(self, n_ctx, n_query, d, seed=None):
        g = torch.Generator()
        if seed is not None:
            g.manual_seed(seed)
        N = n_ctx + n_query
        X = torch.randn(N, d, generator=g)
        eps = self.sigma_eps * torch.randn(N, generator=g)
        y = X @ self.w + eps
        return X[:n_ctx], y[:n_ctx], X[n_ctx:], y[n_ctx:]


torch.manual_seed(99)
fixed_w = torch.randn(D)
narrow_prior = FixedWPrior(w=fixed_w, sigma_eps=0.1)

# Healthy training on the narrow prior.
model = make_model(d=D, seed=42)
narrow_trainer = ICLTrainer(prior=narrow_prior, model=model, n_steps=400,
                            n_ctx=48, n_query=24, d=D, lr=3e-3,
                            warmup_steps=30, grad_clip=1.0, eval_every=40, seed=3)
narrow_diag = narrow_trainer.train()

# Evaluate the narrow-trained model on a broad prior (SCM-MLP).
broad_prior = priors_for_plot['SCM-MLP']
def eval_on(prior, model, n_tasks=20, n_ctx=48, n_q=24):
    model.eval()
    losses = []
    with torch.no_grad():
        for k in range(n_tasks):
            X_c, y_c, X_q, y_q = prior.sample_episode(n_ctx, n_q, D, seed=33333 + k)
            yhat = model(X_q, X_c, y_c)
            v = y_q.var().clamp_min(1e-3)
            losses.append((((yhat - y_q) ** 2).mean() / v).item())
    return float(np.mean(losses)), float(np.std(losses) / (len(losses) ** 0.5))

narrow_on_narrow = eval_on(narrow_prior, model)
narrow_on_broad = eval_on(broad_prior, model)
# Also a healthy model trained on the broad prior, for comparison.
broad_model = make_model(d=D, seed=42)
broad_trainer = ICLTrainer(prior=broad_prior, model=broad_model, n_steps=400,
                           n_ctx=48, n_query=24, d=D, lr=3e-3,
                           warmup_steps=30, grad_clip=1.0, eval_every=0, seed=3)
_ = broad_trainer.train()
broad_on_broad = eval_on(broad_prior, broad_model)
broad_on_narrow = eval_on(narrow_prior, broad_model)

print(f'narrow-trained PFN on narrow eval : {narrow_on_narrow[0]:.3f} +/- {narrow_on_narrow[1]:.3f}')
print(f'narrow-trained PFN on broad  eval : {narrow_on_broad[0]:.3f} +/- {narrow_on_broad[1]:.3f}')
print(f'broad-trained  PFN on narrow eval : {broad_on_narrow[0]:.3f} +/- {broad_on_narrow[1]:.3f}')
print(f'broad-trained  PFN on broad  eval : {broad_on_broad[0]:.3f} +/- {broad_on_broad[1]:.3f}')

fig, ax = plt.subplots(1, 1, figsize=(7, 3.8))
labels = ['narrow→narrow', 'narrow→broad', 'broad→narrow', 'broad→broad']
vals = [narrow_on_narrow[0], narrow_on_broad[0], broad_on_narrow[0], broad_on_broad[0]]
errs = [narrow_on_narrow[1], narrow_on_broad[1], broad_on_narrow[1], broad_on_broad[1]]
colors = ['C3', 'C3', 'C0', 'C0']
ax.bar(np.arange(4), vals, yerr=errs, color=colors, alpha=0.85, capsize=4)
ax.set_xticks(np.arange(4)); ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel('normalised eval MSE')
ax.set_title('Figure 21.3: Over-narrow prior fails to transfer')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_21_03_failure_mode.pdf', bbox_inches='tight')
plt.show()
